# Experiment Data Analysis

This notebook retrieves and analyzes data from MongoDB for experiment `exp_dataset_quality`.

**Contents:**
1. MongoDB Connection & Experiment Config
2. Trajectories Analysis
3. Perturbations Analysis
4. Summary Statistics

In [1]:
# Setup and imports
import os
import sys
from pathlib import Path
from pprint import pprint
from collections import Counter

import pandas as pd
from pymongo import MongoClient
from dotenv import load_dotenv

# Add src to path for project imports
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

# Load environment variables
load_dotenv(project_root / ".env")

print(f"Project root: {project_root}")

Project root: /Users/amanzing/Paper-Research


## 1. MongoDB Connection & Experiment Config

In [2]:
# Connect to MongoDB
MONGODB_URI = os.getenv("MONGODB_URI")
MONGODB_DATABASE = os.getenv("MONGODB_DATABASE", "agent_judge_experiment")

print(f"Database: {MONGODB_DATABASE}")
print(f"URI: {MONGODB_URI[:50]}..." if MONGODB_URI else "URI not found!")

client = MongoClient(MONGODB_URI)
db = client[MONGODB_DATABASE]

# Test connection
client.admin.command('ping')
print("\nConnected to MongoDB successfully!")

Database: agent_judge_experiment
URI: mongodb+srv://singh96aman_db_user:rzVqnZkWrDrsryRy...

Connected to MongoDB successfully!


In [3]:
# List all collections
collections = db.list_collection_names()
print("Available collections:")
for coll in collections:
    count = db[coll].count_documents({})
    print(f"  - {coll}: {count:,} documents")

Available collections:
  - experiments: 2 documents
  - perturbations: 5,804 documents
  - annotations: 26 documents
  - trajectories: 6,454 documents
  - judge_evaluations: 852 documents
  - _connection_test: 0 documents
  - ccg_scores: 0 documents


In [7]:
# Get experiment config
EXPERIMENT_ID = "exp_dataset_quality"

experiment = db.experiments.find_one({"experiment_id": EXPERIMENT_ID})

if experiment:
    print(f"Experiment: {experiment.get('experiment_id')}")
    print(f"Name: {experiment.get('name')}")
    print(f"Description: {experiment.get('description')}")
    print(f"Status: {experiment.get('status')}")
    print(f"Created at: {experiment.get('created_at')}")
else:
    print(f"Experiment '{EXPERIMENT_ID}' not found!")
    # List available experiments
    print("\nAvailable experiments:")
    for exp in db.experiments.find().limit(10):
        print(f"  - {exp.get('experiment_id')}: {exp.get('name')}")

Experiment: exp_dataset_quality
Name: Aman reviewed dataset quality check
Description: Aman reviewed dataset quality check
Status: created
Created at: 2026-04-04 02:55:48.969000


In [8]:
# Show full experiment config
if experiment:
    print("Full experiment document:")
    pprint(experiment)

Full experiment document:
{'_id': ObjectId('69d07db44f620bde9bb54187'),
 'config': {'annotation': {'annotator_id': 'researcher1',
                           'num_samples': 50,
                           'random_seed': 42,
                           'sampling_strategy': 'stratified',
                           'skip_annotated': True},
            'ccg': {'generate_plots': True,
                    'judges': ['claude-sonnet-4.5', 'gpt-oss-120b'],
                    'output_formats': ['csv', 'json']},
            'datasets': {'gaia': {'enabled': True,
                                  'filters': {'max_steps': 100, 'min_steps': 4},
                                  'json_path': 'data/sampled/gaia_100.json',
                                  'num_trajectories': 100,
                                  'random_seed': 42,
                                  'source': 'json'},
                         'swebench': {'enabled': True,
                                      'filters': {'max_steps': 30,

## 2. Trajectories Analysis

In [9]:
# Get all trajectories for this experiment
# First, get trajectory IDs from perturbations collection
perturbations_cursor = db.perturbations.find({"experiment_id": EXPERIMENT_ID})
perturbations_list = list(perturbations_cursor)

print(f"Found {len(perturbations_list)} perturbations for experiment '{EXPERIMENT_ID}'")

# Extract unique trajectory IDs
original_ids = set(p["original_trajectory_id"] for p in perturbations_list if "original_trajectory_id" in p)
perturbed_ids = set(p["perturbed_trajectory_id"] for p in perturbations_list if "perturbed_trajectory_id" in p)

print(f"Unique original trajectories: {len(original_ids)}")
print(f"Unique perturbed trajectories: {len(perturbed_ids)}")

Found 5361 perturbations for experiment 'exp_dataset_quality'
Unique original trajectories: 600
Unique perturbed trajectories: 5361


In [10]:
# Fetch original trajectories
original_trajectories = list(db.trajectories.find(
    {"trajectory_id": {"$in": list(original_ids)}}
))

print(f"Fetched {len(original_trajectories)} original trajectories")

# Show sample
if original_trajectories:
    sample = original_trajectories[0]
    print("\nSample trajectory keys:")
    print(list(sample.keys()))

Fetched 600 original trajectories

Sample trajectory keys:
['_id', 'trajectory_id', 'benchmark', 'steps', 'ground_truth', 'metadata', 'domain', 'complexity', 'experiment_id', 'is_perturbed', 'stored_at']


In [11]:
sample

{'_id': ObjectId('69d07df74f620bde9bb54327'),
 'trajectory_id': 'gaia_1',
 'benchmark': 'gaia',
 'steps': [{'step_id': 'gaia_1_step_1',
   'step_number': 1,
   'step_type': 'tool_execution',
   'content': 'Search the web for “finding nemo main character”.',
   'tool_name': 'web_search',
   'tool_input': None,
   'tool_output': None,
   'metadata': {}},
  {'step_id': 'gaia_1_step_2',
   'step_number': 2,
   'step_type': 'reasoning',
   'content': 'Note the results, which state that the main character is a clownfish.',
   'tool_name': None,
   'tool_input': None,
   'tool_output': None,
   'metadata': {}},
  {'step_id': 'gaia_1_step_3',
   'step_number': 3,
   'step_type': 'tool_execution',
   'content': 'Search the web for “usgs nonnative species database”.',
   'tool_name': 'web_search',
   'tool_input': None,
   'tool_output': None,
   'metadata': {}},
  {'step_id': 'gaia_1_step_4',
   'step_number': 4,
   'step_type': 'tool_execution',
   'content': 'Click result for the Nonindigenou

In [ ]:
# Analyze trajectory distribution by benchmark
benchmark_counts = Counter(t.get("benchmark", "unknown") for t in original_trajectories)

print("Trajectories by benchmark:")
for benchmark, count in sorted(benchmark_counts.items()):
    print(f"  {benchmark}: {count}")

In [ ]:
# Analyze trajectory step counts
step_counts = []
for t in original_trajectories:
    steps = t.get("steps", [])
    step_counts.append(len(steps))

df_steps = pd.DataFrame({"step_count": step_counts})
print("Step count statistics:")
print(df_steps.describe())

In [ ]:
# Create trajectories DataFrame for analysis
trajectories_data = []
for t in original_trajectories:
    trajectories_data.append({
        "trajectory_id": t.get("trajectory_id"),
        "benchmark": t.get("benchmark", "unknown"),
        "task": t.get("task", "")[:100],  # Truncate for display
        "num_steps": len(t.get("steps", [])),
        "domain": t.get("metadata", {}).get("domain", "unknown"),
        "complexity": t.get("metadata", {}).get("complexity", "unknown"),
    })

df_trajectories = pd.DataFrame(trajectories_data)
print(f"Trajectories DataFrame shape: {df_trajectories.shape}")
df_trajectories.head()

In [ ]:
# Show a sample trajectory in detail
if original_trajectories:
    sample_traj = original_trajectories[0]
    print(f"Sample Trajectory: {sample_traj.get('trajectory_id')}")
    print(f"Benchmark: {sample_traj.get('benchmark')}")
    print(f"Task: {sample_traj.get('task', 'N/A')[:200]}...")
    print(f"\nSteps ({len(sample_traj.get('steps', []))}):\n")
    
    for i, step in enumerate(sample_traj.get("steps", [])[:5]):  # Show first 5 steps
        print(f"Step {i+1}:")
        print(f"  Type: {step.get('type', 'N/A')}")
        content = str(step.get('content', step.get('action', 'N/A')))[:150]
        print(f"  Content: {content}...")
        print()

## 3. Perturbations Analysis

In [ ]:
# Create perturbations DataFrame
perturbations_data = []
for p in perturbations_list:
    perturbations_data.append({
        "perturbation_id": p.get("perturbation_id"),
        "original_trajectory_id": p.get("original_trajectory_id"),
        "perturbed_trajectory_id": p.get("perturbed_trajectory_id"),
        "perturbation_type": p.get("perturbation_type"),
        "perturbation_position": p.get("perturbation_position"),
        "step_index": p.get("perturbation_config", {}).get("step_index"),
        "method": p.get("perturbation_config", {}).get("method"),
        "benchmark": p.get("benchmark", "unknown"),
        "created_at": p.get("created_at"),
    })

df_perturbations = pd.DataFrame(perturbations_data)
print(f"Perturbations DataFrame shape: {df_perturbations.shape}")
df_perturbations.head()

In [ ]:
# Perturbation distribution by type
print("Perturbations by type:")
print(df_perturbations["perturbation_type"].value_counts())

In [ ]:
# Perturbation distribution by position
print("Perturbations by position:")
print(df_perturbations["perturbation_position"].value_counts())

In [ ]:
# Cross-tabulation: Type x Position
print("Perturbation distribution (Type x Position):")
crosstab = pd.crosstab(
    df_perturbations["perturbation_type"],
    df_perturbations["perturbation_position"],
    margins=True
)
print(crosstab)

In [ ]:
# Perturbations by benchmark
print("Perturbations by benchmark:")
print(df_perturbations["benchmark"].value_counts())

In [ ]:
# Show a sample perturbation in detail
if perturbations_list:
    sample_pert = perturbations_list[0]
    print("Sample Perturbation:")
    print(f"  ID: {sample_pert.get('perturbation_id')}")
    print(f"  Type: {sample_pert.get('perturbation_type')}")
    print(f"  Position: {sample_pert.get('perturbation_position')}")
    print(f"  Original trajectory: {sample_pert.get('original_trajectory_id')}")
    print(f"  Perturbed trajectory: {sample_pert.get('perturbed_trajectory_id')}")
    print(f"\nConfig:")
    pprint(sample_pert.get('perturbation_config', {}))

In [ ]:
# Fetch a perturbed trajectory to compare with original
if perturbations_list:
    sample_pert = perturbations_list[0]
    
    # Get original
    original = db.trajectories.find_one(
        {"trajectory_id": sample_pert["original_trajectory_id"]}
    )
    
    # Get perturbed
    perturbed = db.trajectories.find_one(
        {"trajectory_id": sample_pert["perturbed_trajectory_id"]}
    )
    
    if original and perturbed:
        step_idx = sample_pert.get("perturbation_config", {}).get("step_index", 0)
        print(f"Comparing step {step_idx} (0-indexed):")
        print(f"\nPerturbation type: {sample_pert.get('perturbation_type')}")
        print(f"Perturbation position: {sample_pert.get('perturbation_position')}")
        print("\n" + "="*60)
        print("ORIGINAL:")
        print("="*60)
        if step_idx < len(original.get("steps", [])):
            pprint(original["steps"][step_idx])
        print("\n" + "="*60)
        print("PERTURBED:")
        print("="*60)
        if step_idx < len(perturbed.get("steps", [])):
            pprint(perturbed["steps"][step_idx])

## 4. Summary Statistics

In [ ]:
# Overall summary
print("="*60)
print(f"EXPERIMENT SUMMARY: {EXPERIMENT_ID}")
print("="*60)
print(f"\nTrajectories:")
print(f"  Total original: {len(original_trajectories)}")
print(f"  Total perturbed: {len(perturbed_ids)}")

print(f"\nPerturbations:")
print(f"  Total: {len(perturbations_list)}")

print(f"\nBy Type:")
for ptype, count in df_perturbations["perturbation_type"].value_counts().items():
    print(f"  {ptype}: {count}")

print(f"\nBy Position:")
for pos, count in df_perturbations["perturbation_position"].value_counts().items():
    print(f"  {pos}: {count}")

print(f"\nBy Benchmark:")
for bench, count in df_perturbations["benchmark"].value_counts().items():
    print(f"  {bench}: {count}")

In [ ]:
# Check for judge evaluations (if any)
eval_count = db.judge_evaluations.count_documents({"experiment_id": EXPERIMENT_ID})
print(f"Judge evaluations for this experiment: {eval_count}")

if eval_count > 0:
    # Get sample evaluation
    sample_eval = db.judge_evaluations.find_one({"experiment_id": EXPERIMENT_ID})
    print("\nSample evaluation:")
    pprint(sample_eval)

In [ ]:
# Close connection
client.close()
print("MongoDB connection closed.")